In [1]:
import sys
!{sys.executable} -m pip install -q numpy gymnasium matplotlib pandas stable-baselines3 scipy
print("All packages installed.")

All packages installed.


In [2]:
import numpy as np
from dataclasses import dataclass

FIBONACCI_POINTS = [1, 2, 3, 5, 8, 13]
MAX_EFFORT = max(FIBONACCI_POINTS)

ITEM_NAME_POOL = [
    "Bulk CSV export", "SSO login (SAML)", "Dark mode UI", "Async job queue",
    "Slack integration", "Custom dashboard widgets", "Audit log export",
    "Two-factor authentication", "Webhook retries", "API rate limiting",
    "Kanban swimlanes", "Time-tracking report", "Mobile push notifications",
    "Bulk item archive", "Custom fields for tickets", "SLA breach alerts",
    "Guest viewer role", "Recurring task templates", "Dependency graph view",
    "CSV import validation", "Public roadmap page", "Sprint burndown chart",
    "Email digest summaries", "Attachment virus scanning", "Search autocomplete",
    "Bulk reassign owner", "Custom status workflows", "Team capacity planner",
    "Item history timeline", "Keyboard shortcuts", "Offline mode (PWA)",
    "Multi-currency billing", "Usage analytics dashboard", "SSO for SCIM provisioning",
    "Comment @mentions", "Item templates library", "Read-only API tokens",
    "Data residency (EU region)", "Custom SLA policies", "Bulk label editor",
]

@dataclass
class BacklogItem:
    item_id: int
    name: str
    business_value: float
    effort_raw: int
    effort: float
    risk: float
    dependency_count: int
    demand_signal: float
    urgency: float
    age: int = 0

    def to_feature_vector(self):
        return np.array([
            self.business_value, self.effort, self.risk,
            min(self.dependency_count, 3) / 3.0, self.demand_signal,
            self.urgency, min(self.age, 20) / 20.0,
        ], dtype=np.float32)

class BacklogGenerator:
    def __init__(self, seed=None):
        self.rng = np.random.default_rng(seed)
        self._next_id = 0
        self._name_cycle = list(ITEM_NAME_POOL)
        self.rng.shuffle(self._name_cycle)
        self._name_idx = 0

    def _next_name(self):
        name = self._name_cycle[self._name_idx % len(self._name_cycle)]
        self._name_idx += 1
        return name

    def generate_item(self):
        item_id = self._next_id
        self._next_id += 1
        business_value = float(self.rng.beta(2, 2))
        effort_raw = int(self.rng.choice(FIBONACCI_POINTS))
        effort = effort_raw / MAX_EFFORT
        risk = float(self.rng.beta(1.5, 3))
        dependency_count = int(min(self.rng.poisson(0.6), 3))
        demand_signal = float(self.rng.beta(2, 3))
        urgency = float(self.rng.beta(1.5, 4))
        return BacklogItem(
            item_id=item_id, name=self._next_name(),
            business_value=round(business_value, 3), effort_raw=effort_raw,
            effort=round(effort, 3), risk=round(risk, 3),
            dependency_count=dependency_count, demand_signal=round(demand_signal, 3),
            urgency=round(urgency, 3), age=0,
        )

    def generate_backlog(self, n_items):
        return [self.generate_item() for _ in range(n_items)]

print("BacklogGenerator defined.")

BacklogGenerator defined.


In [3]:
import gymnasium as gym
from gymnasium import spaces

MAX_ITEMS = 40
N_FEATURES = 7
EPISODE_LENGTH = 12
SHOCK_PROBABILITY = 0.20

W1_VALUE = 1.0
W2_DEPENDENCY = 0.5
W3_CAPACITY = 1.0
W4_URGENCY = 0.3
URGENCY_THRESHOLD = 0.7

class BacklogPrioritisationEnv(gym.Env):
    metadata = {"render_modes": ["human"]}

    def __init__(self, n_items=30, capacity_points=20, seed=None):
        super().__init__()
        self.n_items_target = n_items
        self.capacity_points = capacity_points
        self.generator = BacklogGenerator(seed=seed)
        obs_dim = MAX_ITEMS * N_FEATURES + 3
        self.observation_space = spaces.Box(low=0.0, high=1.0, shape=(obs_dim,), dtype=np.float32)
        self.action_space = spaces.MultiDiscrete([3] * MAX_ITEMS)
        self.items = []
        self.cycle = 0
        self.last_shock = 0

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        if seed is not None:
            self.generator = BacklogGenerator(seed=seed)
        self.items = self.generator.generate_backlog(self.n_items_target)
        self.cycle = 0
        self.last_shock = 0
        return self._get_obs(), self._get_info()

    def step(self, action):
        shock_applied = self._maybe_apply_shock()
        reward, info = self._compute_reward_and_advance(action)
        self.cycle += 1
        terminated = False
        truncated = self.cycle >= EPISODE_LENGTH
        info["shock_applied"] = shock_applied
        info["cycle"] = self.cycle
        return self._get_obs(), reward, terminated, truncated, info

    def _maybe_apply_shock(self):
        if self.generator.rng.random() >= SHOCK_PROBABILITY or not self.items:
            self.last_shock = 0
            return None
        self.last_shock = 1
        shock_type = self.generator.rng.choice(["demand_spike", "dependency_resolved", "capacity_change"])
        target = self.items[self.generator.rng.integers(0, len(self.items))]
        if shock_type == "demand_spike":
            target.demand_signal = min(1.0, target.demand_signal + 0.4)
        elif shock_type == "dependency_resolved":
            target.dependency_count = max(0, target.dependency_count - 1)
        elif shock_type == "capacity_change":
            delta = self.generator.rng.choice([-5, 5])
            self.capacity_points = max(5, self.capacity_points + delta)
        return shock_type

    def _compute_reward_and_advance(self, action):
        value_delivered = 0.0
        dependency_violation = 0
        urgency_delay = 0.0
        now_effort = 0
        now_items, next_items, later_items = [], [], []

        for idx, item in enumerate(self.items):
            tier = int(action[idx]) if idx < len(action) else 0
            if tier == 2:
                now_items.append(item)
                value_delivered += item.business_value
                now_effort += item.effort_raw
                if item.dependency_count > 0:
                    dependency_violation += 1
            elif tier == 1:
                next_items.append(item)
            else:
                later_items.append(item)

        capacity_overrun = max(0, now_effort - self.capacity_points)

        for item in self.items:
            if item.urgency > URGENCY_THRESHOLD and item not in now_items:
                urgency_delay += item.urgency

        reward = (W1_VALUE * value_delivered - W2_DEPENDENCY * dependency_violation
                  - W3_CAPACITY * capacity_overrun - W4_URGENCY * urgency_delay)

        completed_ids = {it.item_id for it in now_items}
        self.items = [it for it in self.items if it.item_id not in completed_ids]
        for it in self.items:
            it.age += 1
        n_new = max(0, self.n_items_target - len(self.items))
        self.items.extend(self.generator.generate_backlog(n_new))

        info = {"value_delivered": value_delivered, "dependency_violation": dependency_violation,
                "capacity_overrun": capacity_overrun, "urgency_delay": urgency_delay,
                "n_now": len(now_items), "n_next": len(next_items), "n_later": len(later_items)}
        return reward, info

    def _get_obs(self):
        mat = np.zeros((MAX_ITEMS, N_FEATURES), dtype=np.float32)
        for i, item in enumerate(self.items[:MAX_ITEMS]):
            mat[i] = item.to_feature_vector()
        global_feats = np.array([min(self.capacity_points, 50) / 50.0,
                                  min(self.cycle, EPISODE_LENGTH) / EPISODE_LENGTH,
                                  float(self.last_shock)], dtype=np.float32)
        return np.concatenate([mat.flatten(), global_feats])

    def _get_info(self):
        return {"n_active_items": len(self.items), "capacity_points": self.capacity_points}

print("BacklogPrioritisationEnv defined (corrected reward, shocks, replenishment).")

BacklogPrioritisationEnv defined (corrected reward, shocks, replenishment).


In [4]:
from enum import IntEnum

class Tier(IntEnum):
    LATER = 0
    NEXT = 1
    NOW = 2

def rice_score(item, confidence_risk_threshold=0.7):
    reach = item.demand_signal
    impact = item.business_value
    confidence = 0.5 if item.risk > confidence_risk_threshold else 0.8
    effort = max(item.effort, 1e-6)
    return (reach * impact * confidence) / effort

def wsjf_score(item):
    cost_of_delay = item.business_value + item.urgency + (1.0 - item.risk)
    duration = max(item.effort, 1e-6)
    return cost_of_delay / duration

def assign_tiers(items, score_fn, capacity_points, next_band_fraction=0.5):
    scored = sorted(((it, score_fn(it)) for it in items), key=lambda x: x[1], reverse=True)
    tiers = {}
    used_now = 0
    used_next = 0
    next_budget = capacity_points * next_band_fraction
    for item, score in scored:
        if used_now + item.effort_raw <= capacity_points:
            tiers[item.item_id] = Tier.NOW
            used_now += item.effort_raw
        elif used_next + item.effort_raw <= next_budget:
            tiers[item.item_id] = Tier.NEXT
            used_next += item.effort_raw
        else:
            tiers[item.item_id] = Tier.LATER
    return tiers, scored

def policy_action_from_tiers(env, tiers_dict):
    action = np.zeros(MAX_ITEMS, dtype=int)
    for idx, item in enumerate(env.items):
        action[idx] = int(tiers_dict.get(item.item_id, 0))
    return action

print("RICE/WSJF scorers defined.")

RICE/WSJF scorers defined.


In [5]:
NEW_TEST_SEEDS = list(range(200, 215))  # 15 fresh seeds, never touched before
N_CYCLES = 12
CAPACITY = 20

def run_episode_static(policy_name, seed):
    env = BacklogPrioritisationEnv(n_items=30, capacity_points=CAPACITY, seed=seed)
    obs, info = env.reset(seed=seed)
    rewards = []
    for cycle in range(N_CYCLES):
        if policy_name == "random":
            action = env.action_space.sample()
        else:
            score_fn = rice_score if policy_name == "rice" else wsjf_score
            tiers, _ = assign_tiers(env.items, score_fn, env.capacity_points)
            action = policy_action_from_tiers(env, tiers)
        obs, reward, term, trunc, info = env.step(action)
        rewards.append(reward)
        if term or trunc:
            break
    return rewards

baseline_results = {"random": [], "rice": [], "wsjf": []}
for seed in NEW_TEST_SEEDS:
    for policy in baseline_results:
        baseline_results[policy].append(sum(run_episode_static(policy, seed)))

print(f"{'Policy':10s} {'Mean CumReward':>15s} {'Std':>8s}")
for p in ["random", "rice", "wsjf"]:
    print(f"{p:10s} {np.mean(baseline_results[p]):15.2f} {np.std(baseline_results[p]):8.2f}")

Policy      Mean CumReward      Std
random             -368.38    53.65
rice                 18.26     7.63
wsjf                 16.59     6.32


In [6]:
import os, time
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import BaseCallback

TRAIN_SEEDS = [10, 11, 12, 13, 14, 15, 16, 17]  # 8 fresh seeds, per frozen protocol
TOTAL_TIMESTEPS_PER_SEED = 98_304
MODEL_DIR = "models"
os.makedirs(MODEL_DIR, exist_ok=True)

def linear_schedule(initial_value, final_value):
    def schedule(progress_remaining):
        return final_value + progress_remaining * (initial_value - final_value)
    return schedule

class ConvergenceLogger(BaseCallback):
    def __init__(self, log_every=2048, verbose=0):
        super().__init__(verbose)
        self.log_every = log_every
        self.history = []
    def _on_step(self):
        if self.n_calls % self.log_every == 0:
            buf = self.model.ep_info_buffer
            if buf and len(buf) > 0:
                mean_reward = float(np.mean([ep["r"] for ep in buf]))
                self.history.append((self.num_timesteps, mean_reward))
                print(f"  seed {seed} | timestep={self.num_timesteps:>7d} | mean_ep_reward={mean_reward:8.2f}")
        return True

overall_start = time.time()
for seed in TRAIN_SEEDS:
    print(f"\n=== Training seed {seed} ({TRAIN_SEEDS.index(seed)+1}/{len(TRAIN_SEEDS)}) ===")
    env = Monitor(BacklogPrioritisationEnv(n_items=30, capacity_points=20, seed=seed))
    model = PPO(
        policy="MlpPolicy", env=env,
        learning_rate=linear_schedule(3e-4, 3e-5),
        n_steps=2048, batch_size=64, gamma=0.99, clip_range=0.2,
        policy_kwargs=dict(net_arch=dict(pi=[128, 128], vf=[128, 128])),
        verbose=0, seed=seed, device="cpu",
    )
    logger_cb = ConvergenceLogger(log_every=2048)
    start = time.time()
    model.learn(total_timesteps=TOTAL_TIMESTEPS_PER_SEED, callback=logger_cb, progress_bar=False)
    elapsed = time.time() - start
    model.save(f"{MODEL_DIR}/ppo_seed{seed}")
    np.save(f"{MODEL_DIR}/convergence_seed{seed}.npy", np.array(logger_cb.history))
    print(f"Seed {seed} complete in {elapsed/60:.1f} minutes.")

total_elapsed = time.time() - overall_start
print(f"\nAll 8 seeds trained. Total time: {total_elapsed/60:.1f} minutes.")


=== Training seed 10 (1/8) ===
  seed 10 | timestep=   2048 | mean_ep_reward= -310.78
  seed 10 | timestep=   4096 | mean_ep_reward= -166.35
  seed 10 | timestep=   6144 | mean_ep_reward=   30.57
  seed 10 | timestep=   8192 | mean_ep_reward=   21.53
  seed 10 | timestep=  10240 | mean_ep_reward=  -10.17
  seed 10 | timestep=  12288 | mean_ep_reward=   21.02
  seed 10 | timestep=  14336 | mean_ep_reward=   14.05
  seed 10 | timestep=  16384 | mean_ep_reward=   30.89
  seed 10 | timestep=  18432 | mean_ep_reward=   32.26
  seed 10 | timestep=  20480 | mean_ep_reward=   32.87
  seed 10 | timestep=  22528 | mean_ep_reward=   32.72
  seed 10 | timestep=  24576 | mean_ep_reward=   33.95
  seed 10 | timestep=  26624 | mean_ep_reward=   34.83
  seed 10 | timestep=  28672 | mean_ep_reward=   34.45
  seed 10 | timestep=  30720 | mean_ep_reward=   36.58
  seed 10 | timestep=  32768 | mean_ep_reward=   33.36
  seed 10 | timestep=  34816 | mean_ep_reward=   27.96
  seed 10 | timestep=  36864 | me

In [1]:
from scipy import stats
from stable_baselines3 import PPO

def run_episode_ppo(model, seed):
    env = BacklogPrioritisationEnv(n_items=30, capacity_points=CAPACITY, seed=seed)
    obs, info = env.reset(seed=seed)
    rewards = []
    for cycle in range(N_CYCLES):
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, term, trunc, info = env.step(action)
        rewards.append(reward)
        if term or trunc:
            break
    return rewards

def run_episode_ppo_stochastic(model, seed):
    env = BacklogPrioritisationEnv(n_items=30, capacity_points=CAPACITY, seed=seed)
    obs, info = env.reset(seed=seed)
    rewards = []
    for cycle in range(N_CYCLES):
        action, _ = model.predict(obs, deterministic=False)
        obs, reward, term, trunc, info = env.step(action)
        rewards.append(reward)
        if term or trunc:
            break
    return rewards

ppo_models = {s: PPO.load(f"{MODEL_DIR}/ppo_seed{s}", device="cpu") for s in TRAIN_SEEDS}

# --- Primary unit of analysis: one mean-per-training-seed, across all 15 test scenarios ---
seed_level_means_det = []
seed_level_means_stoch = []
for s in TRAIN_SEEDS:
    det_totals = [sum(run_episode_ppo(ppo_models[s], ts)) for ts in NEW_TEST_SEEDS]
    stoch_totals = [sum(run_episode_ppo_stochastic(ppo_models[s], ts)) for ts in NEW_TEST_SEEDS]
    seed_level_means_det.append(np.mean(det_totals))
    seed_level_means_stoch.append(np.mean(stoch_totals))
    print(f"Seed {s}: deterministic mean = {np.mean(det_totals):8.2f} | stochastic mean = {np.mean(stoch_totals):8.2f}")

print(f"\n{'='*70}")
print("PRIMARY RESULT: 8 seed-level means (deterministic)")
print(f"{'='*70}")
print(f"Values: {[f'{x:.2f}' for x in seed_level_means_det]}")
print(f"Mean: {np.mean(seed_level_means_det):.2f}  Median: {np.median(seed_level_means_det):.2f}")
print(f"Std: {np.std(seed_level_means_det):.2f}  IQR: {np.percentile(seed_level_means_det,75)-np.percentile(seed_level_means_det,25):.2f}")

# --- Assumption check ---
shapiro_stat, shapiro_p = stats.shapiro(seed_level_means_det)
print(f"\nShapiro-Wilk normality test: statistic={shapiro_stat:.4f}, p={shapiro_p:.4f}")
print("-> Data is", "approximately normal" if shapiro_p > 0.05 else "NOT normal (non-parametric test is appropriate)")

# --- Primary test: Wilcoxon signed-rank, RICE and WSJF as fixed reference values per test scenario ---
rice_per_scenario = baseline_results["rice"]
wsjf_per_scenario = baseline_results["wsjf"]

# Wilcoxon needs paired data of equal length; compare each of the 8 seed means against the
# scenario-level RICE/WSJF distribution's median as the reference point (one-sample form)
rice_median = np.median(rice_per_scenario)
wsjf_median = np.median(wsjf_per_scenario)

w_rice, p_rice = stats.wilcoxon(np.array(seed_level_means_det) - rice_median)
w_wsjf, p_wsjf = stats.wilcoxon(np.array(seed_level_means_det) - wsjf_median)

# Holm-Bonferroni correction across 2 comparisons
raw_pvals = [p_rice, p_wsjf]
sorted_idx = np.argsort(raw_pvals)
holm_adjusted = [0, 0]
for rank, idx in enumerate(sorted_idx):
    holm_adjusted[idx] = min(raw_pvals[idx] * (len(raw_pvals) - rank), 1.0)

print(f"\n--- Wilcoxon signed-rank test (primary, pre-registered) ---")
print(f"vs RICE (median {rice_median:.2f}): W={w_rice:.3f}, raw p={p_rice:.4f}, Holm-adjusted p={holm_adjusted[0]:.4f}")
print(f"vs WSJF (median {wsjf_median:.2f}): W={w_wsjf:.3f}, raw p={p_wsjf:.4f}, Holm-adjusted p={holm_adjusted[1]:.4f}")

# --- Bootstrap 95% CI on median difference ---
rng = np.random.default_rng(42)
def bootstrap_median_diff(seed_means, reference_median, n_boot=10000):
    diffs = []
    for _ in range(n_boot):
        sample = rng.choice(seed_means, size=len(seed_means), replace=True)
        diffs.append(np.median(sample) - reference_median)
    return np.percentile(diffs, 2.5), np.percentile(diffs, 97.5)

ci_rice = bootstrap_median_diff(seed_level_means_det, rice_median)
ci_wsjf = bootstrap_median_diff(seed_level_means_det, wsjf_median)
print(f"\nBootstrap 95% CI of median difference vs RICE: [{ci_rice[0]:.2f}, {ci_rice[1]:.2f}]")
print(f"Bootstrap 95% CI of median difference vs WSJF: [{ci_wsjf[0]:.2f}, {ci_wsjf[1]:.2f}]")

# --- Practical equivalence check (+/- 5.0 margin) ---
EQUIVALENCE_MARGIN = 5.0
median_diff_rice = np.median(seed_level_means_det) - rice_median
median_diff_wsjf = np.median(seed_level_means_det) - wsjf_median
print(f"\nMedian difference vs RICE: {median_diff_rice:.2f} -> {'WITHIN' if abs(median_diff_rice) <= EQUIVALENCE_MARGIN else 'OUTSIDE'} equivalence margin (+/-{EQUIVALENCE_MARGIN})")
print(f"Median difference vs WSJF: {median_diff_wsjf:.2f} -> {'WITHIN' if abs(median_diff_wsjf) <= EQUIVALENCE_MARGIN else 'OUTSIDE'} equivalence margin (+/-{EQUIVALENCE_MARGIN})")

np.savez("frozen_protocol_results.npz",
         train_seeds=TRAIN_SEEDS, test_seeds=NEW_TEST_SEEDS,
         seed_level_means_det=seed_level_means_det, seed_level_means_stoch=seed_level_means_stoch,
         rice_per_scenario=rice_per_scenario, wsjf_per_scenario=wsjf_per_scenario,
         random_per_scenario=baseline_results["random"])
print("\nSaved frozen_protocol_results.npz")

NameError: name 'TRAIN_SEEDS' is not defined

In [4]:
import numpy as np
import gymnasium as gym
from gymnasium import spaces
from dataclasses import dataclass
from enum import IntEnum
from stable_baselines3 import PPO
from scipy import stats

FIBONACCI_POINTS = [1, 2, 3, 5, 8, 13]
MAX_EFFORT = max(FIBONACCI_POINTS)
ITEM_NAME_POOL = ["Bulk CSV export", "SSO login (SAML)", "Dark mode UI", "Async job queue",
    "Slack integration", "Custom dashboard widgets", "Audit log export", "Two-factor authentication",
    "Webhook retries", "API rate limiting", "Kanban swimlanes", "Time-tracking report",
    "Mobile push notifications", "Bulk item archive", "Custom fields for tickets", "SLA breach alerts",
    "Guest viewer role", "Recurring task templates", "Dependency graph view", "CSV import validation",
    "Public roadmap page", "Sprint burndown chart", "Email digest summaries", "Attachment virus scanning",
    "Search autocomplete", "Bulk reassign owner", "Custom status workflows", "Team capacity planner",
    "Item history timeline", "Keyboard shortcuts", "Offline mode (PWA)", "Multi-currency billing",
    "Usage analytics dashboard", "SSO for SCIM provisioning", "Comment @mentions", "Item templates library",
    "Read-only API tokens", "Data residency (EU region)", "Custom SLA policies", "Bulk label editor"]

@dataclass
class BacklogItem:
    item_id: int; name: str; business_value: float; effort_raw: int; effort: float
    risk: float; dependency_count: int; demand_signal: float; urgency: float; age: int = 0
    def to_feature_vector(self):
        return np.array([self.business_value, self.effort, self.risk, min(self.dependency_count, 3)/3.0,
                          self.demand_signal, self.urgency, min(self.age, 20)/20.0], dtype=np.float32)

class BacklogGenerator:
    def __init__(self, seed=None):
        self.rng = np.random.default_rng(seed); self._next_id = 0
        self._name_cycle = list(ITEM_NAME_POOL); self.rng.shuffle(self._name_cycle); self._name_idx = 0
    def _next_name(self):
        name = self._name_cycle[self._name_idx % len(self._name_cycle)]; self._name_idx += 1; return name
    def generate_item(self):
        item_id = self._next_id; self._next_id += 1
        business_value = float(self.rng.beta(2, 2)); effort_raw = int(self.rng.choice(FIBONACCI_POINTS))
        effort = effort_raw / MAX_EFFORT; risk = float(self.rng.beta(1.5, 3))
        dependency_count = int(min(self.rng.poisson(0.6), 3)); demand_signal = float(self.rng.beta(2, 3))
        urgency = float(self.rng.beta(1.5, 4))
        return BacklogItem(item_id=item_id, name=self._next_name(), business_value=round(business_value,3),
            effort_raw=effort_raw, effort=round(effort,3), risk=round(risk,3),
            dependency_count=dependency_count, demand_signal=round(demand_signal,3),
            urgency=round(urgency,3), age=0)
    def generate_backlog(self, n_items):
        return [self.generate_item() for _ in range(n_items)]

MAX_ITEMS = 40; N_FEATURES = 7; EPISODE_LENGTH = 12; SHOCK_PROBABILITY = 0.20
W1_VALUE = 1.0; W2_DEPENDENCY = 0.5; W3_CAPACITY = 1.0; W4_URGENCY = 0.3; URGENCY_THRESHOLD = 0.7

class BacklogPrioritisationEnv(gym.Env):
    metadata = {"render_modes": ["human"]}
    def __init__(self, n_items=30, capacity_points=20, seed=None):
        super().__init__()
        self.n_items_target = n_items; self.capacity_points = capacity_points
        self.generator = BacklogGenerator(seed=seed)
        obs_dim = MAX_ITEMS * N_FEATURES + 3
        self.observation_space = spaces.Box(low=0.0, high=1.0, shape=(obs_dim,), dtype=np.float32)
        self.action_space = spaces.MultiDiscrete([3] * MAX_ITEMS)
        self.items = []; self.cycle = 0; self.last_shock = 0
    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        if seed is not None: self.generator = BacklogGenerator(seed=seed)
        self.items = self.generator.generate_backlog(self.n_items_target)
        self.cycle = 0; self.last_shock = 0
        return self._get_obs(), self._get_info()
    def step(self, action):
        shock_applied = self._maybe_apply_shock()
        reward, info = self._compute_reward_and_advance(action)
        self.cycle += 1; terminated = False; truncated = self.cycle >= EPISODE_LENGTH
        info["shock_applied"] = shock_applied; info["cycle"] = self.cycle
        return self._get_obs(), reward, terminated, truncated, info
    def _maybe_apply_shock(self):
        if self.generator.rng.random() >= SHOCK_PROBABILITY or not self.items:
            self.last_shock = 0; return None
        self.last_shock = 1
        shock_type = self.generator.rng.choice(["demand_spike", "dependency_resolved", "capacity_change"])
        target = self.items[self.generator.rng.integers(0, len(self.items))]
        if shock_type == "demand_spike": target.demand_signal = min(1.0, target.demand_signal + 0.4)
        elif shock_type == "dependency_resolved": target.dependency_count = max(0, target.dependency_count - 1)
        elif shock_type == "capacity_change":
            delta = self.generator.rng.choice([-5, 5]); self.capacity_points = max(5, self.capacity_points + delta)
        return shock_type
    def _compute_reward_and_advance(self, action):
        value_delivered = 0.0; dependency_violation = 0; urgency_delay = 0.0; now_effort = 0
        now_items, next_items, later_items = [], [], []
        for idx, item in enumerate(self.items):
            tier = int(action[idx]) if idx < len(action) else 0
            if tier == 2:
                now_items.append(item); value_delivered += item.business_value; now_effort += item.effort_raw
                if item.dependency_count > 0: dependency_violation += 1
            elif tier == 1: next_items.append(item)
            else: later_items.append(item)
        capacity_overrun = max(0, now_effort - self.capacity_points)
        for item in self.items:
            if item.urgency > URGENCY_THRESHOLD and item not in now_items: urgency_delay += item.urgency
        reward = (W1_VALUE*value_delivered - W2_DEPENDENCY*dependency_violation
                  - W3_CAPACITY*capacity_overrun - W4_URGENCY*urgency_delay)
        completed_ids = {it.item_id for it in now_items}
        self.items = [it for it in self.items if it.item_id not in completed_ids]
        for it in self.items: it.age += 1
        n_new = max(0, self.n_items_target - len(self.items))
        self.items.extend(self.generator.generate_backlog(n_new))
        info = {"value_delivered": value_delivered, "dependency_violation": dependency_violation,
                "capacity_overrun": capacity_overrun, "urgency_delay": urgency_delay,
                "n_now": len(now_items), "n_next": len(next_items), "n_later": len(later_items)}
        return reward, info
    def _get_obs(self):
        mat = np.zeros((MAX_ITEMS, N_FEATURES), dtype=np.float32)
        for i, item in enumerate(self.items[:MAX_ITEMS]): mat[i] = item.to_feature_vector()
        global_feats = np.array([min(self.capacity_points, 50)/50.0, min(self.cycle, EPISODE_LENGTH)/EPISODE_LENGTH,
                                  float(self.last_shock)], dtype=np.float32)
        return np.concatenate([mat.flatten(), global_feats])
    def _get_info(self):
        return {"n_active_items": len(self.items), "capacity_points": self.capacity_points}

class Tier(IntEnum):
    LATER = 0; NEXT = 1; NOW = 2

def rice_score(item, confidence_risk_threshold=0.7):
    reach = item.demand_signal; impact = item.business_value
    confidence = 0.5 if item.risk > confidence_risk_threshold else 0.8
    effort = max(item.effort, 1e-6)
    return (reach * impact * confidence) / effort

def wsjf_score(item):
    cost_of_delay = item.business_value + item.urgency + (1.0 - item.risk)
    duration = max(item.effort, 1e-6)
    return cost_of_delay / duration

def assign_tiers(items, score_fn, capacity_points, next_band_fraction=0.5):
    scored = sorted(((it, score_fn(it)) for it in items), key=lambda x: x[1], reverse=True)
    tiers = {}; used_now = 0; used_next = 0; next_budget = capacity_points * next_band_fraction
    for item, score in scored:
        if used_now + item.effort_raw <= capacity_points:
            tiers[item.item_id] = Tier.NOW; used_now += item.effort_raw
        elif used_next + item.effort_raw <= next_budget:
            tiers[item.item_id] = Tier.NEXT; used_next += item.effort_raw
        else: tiers[item.item_id] = Tier.LATER
    return tiers, scored

def policy_action_from_tiers(env, tiers_dict):
    action = np.zeros(MAX_ITEMS, dtype=int)
    for idx, item in enumerate(env.items): action[idx] = int(tiers_dict.get(item.item_id, 0))
    return action

TRAIN_SEEDS = [10, 11, 12, 13, 14, 15, 16, 17]
NEW_TEST_SEEDS = list(range(200, 215))
N_CYCLES = 12; CAPACITY = 20; MODEL_DIR = "models"

def run_episode_static(policy_name, seed):
    env = BacklogPrioritisationEnv(n_items=30, capacity_points=CAPACITY, seed=seed)
    obs, info = env.reset(seed=seed)
    rewards = []
    for cycle in range(N_CYCLES):
        if policy_name == "random": action = env.action_space.sample()
        else:
            score_fn = rice_score if policy_name == "rice" else wsjf_score
            tiers, _ = assign_tiers(env.items, score_fn, env.capacity_points)
            action = policy_action_from_tiers(env, tiers)
        obs, reward, term, trunc, info = env.step(action)
        rewards.append(reward)
        if term or trunc: break
    return rewards

baseline_results = {"random": [], "rice": [], "wsjf": []}
for seed in NEW_TEST_SEEDS:
    for policy in baseline_results:
        baseline_results[policy].append(sum(run_episode_static(policy, seed)))

print("Reloaded successfully.")
print(f"RICE: {np.mean(baseline_results['rice']):.2f}  WSJF: {np.mean(baseline_results['wsjf']):.2f}")

import os
missing = [s for s in TRAIN_SEEDS if not os.path.exists(f"{MODEL_DIR}/ppo_seed{s}.zip")]
if missing:
    print(f"WARNING: missing model files for seeds {missing}")
else:
    print(f"All 8 trained models found on disk. Ready for Cell 7 evaluation.")

Reloaded successfully.
RICE: 18.26  WSJF: 16.59
All 8 trained models found on disk. Ready for Cell 7 evaluation.


In [3]:
def run_episode_ppo(model, seed):
    env = BacklogPrioritisationEnv(n_items=30, capacity_points=CAPACITY, seed=seed)
    obs, info = env.reset(seed=seed)
    rewards = []
    for cycle in range(N_CYCLES):
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, term, trunc, info = env.step(action)
        rewards.append(reward)
        if term or trunc:
            break
    return rewards

def run_episode_ppo_stochastic(model, seed):
    env = BacklogPrioritisationEnv(n_items=30, capacity_points=CAPACITY, seed=seed)
    obs, info = env.reset(seed=seed)
    rewards = []
    for cycle in range(N_CYCLES):
        action, _ = model.predict(obs, deterministic=False)
        obs, reward, term, trunc, info = env.step(action)
        rewards.append(reward)
        if term or trunc:
            break
    return rewards

ppo_models = {s: PPO.load(f"{MODEL_DIR}/ppo_seed{s}", device="cpu") for s in TRAIN_SEEDS}

seed_level_means_det = []
seed_level_means_stoch = []
for s in TRAIN_SEEDS:
    det_totals = [sum(run_episode_ppo(ppo_models[s], ts)) for ts in NEW_TEST_SEEDS]
    stoch_totals = [sum(run_episode_ppo_stochastic(ppo_models[s], ts)) for ts in NEW_TEST_SEEDS]
    seed_level_means_det.append(np.mean(det_totals))
    seed_level_means_stoch.append(np.mean(stoch_totals))
    print(f"Seed {s}: deterministic mean = {np.mean(det_totals):8.2f} | stochastic mean = {np.mean(stoch_totals):8.2f}")

print(f"\n{'='*70}")
print("PRIMARY RESULT: 8 seed-level means (deterministic)")
print(f"{'='*70}")
print(f"Values: {[f'{x:.2f}' for x in seed_level_means_det]}")
print(f"Mean: {np.mean(seed_level_means_det):.2f}  Median: {np.median(seed_level_means_det):.2f}")
print(f"Std: {np.std(seed_level_means_det):.2f}  IQR: {np.percentile(seed_level_means_det,75)-np.percentile(seed_level_means_det,25):.2f}")

shapiro_stat, shapiro_p = stats.shapiro(seed_level_means_det)
print(f"\nShapiro-Wilk normality test: statistic={shapiro_stat:.4f}, p={shapiro_p:.4f}")
print("-> Data is", "approximately normal" if shapiro_p > 0.05 else "NOT normal (non-parametric test is appropriate)")

rice_per_scenario = baseline_results["rice"]
wsjf_per_scenario = baseline_results["wsjf"]
rice_median = np.median(rice_per_scenario)
wsjf_median = np.median(wsjf_per_scenario)

w_rice, p_rice = stats.wilcoxon(np.array(seed_level_means_det) - rice_median)
w_wsjf, p_wsjf = stats.wilcoxon(np.array(seed_level_means_det) - wsjf_median)

raw_pvals = [p_rice, p_wsjf]
sorted_idx = np.argsort(raw_pvals)
holm_adjusted = [0, 0]
for rank, idx in enumerate(sorted_idx):
    holm_adjusted[idx] = min(raw_pvals[idx] * (len(raw_pvals) - rank), 1.0)

print(f"\n--- Wilcoxon signed-rank test (primary, pre-registered) ---")
print(f"vs RICE (median {rice_median:.2f}): W={w_rice:.3f}, raw p={p_rice:.4f}, Holm-adjusted p={holm_adjusted[0]:.4f}")
print(f"vs WSJF (median {wsjf_median:.2f}): W={w_wsjf:.3f}, raw p={p_wsjf:.4f}, Holm-adjusted p={holm_adjusted[1]:.4f}")

rng = np.random.default_rng(42)
def bootstrap_median_diff(seed_means, reference_median, n_boot=10000):
    diffs = []
    for _ in range(n_boot):
        sample = rng.choice(seed_means, size=len(seed_means), replace=True)
        diffs.append(np.median(sample) - reference_median)
    return np.percentile(diffs, 2.5), np.percentile(diffs, 97.5)

ci_rice = bootstrap_median_diff(seed_level_means_det, rice_median)
ci_wsjf = bootstrap_median_diff(seed_level_means_det, wsjf_median)
print(f"\nBootstrap 95% CI of median difference vs RICE: [{ci_rice[0]:.2f}, {ci_rice[1]:.2f}]")
print(f"Bootstrap 95% CI of median difference vs WSJF: [{ci_wsjf[0]:.2f}, {ci_wsjf[1]:.2f}]")

EQUIVALENCE_MARGIN = 5.0
median_diff_rice = np.median(seed_level_means_det) - rice_median
median_diff_wsjf = np.median(seed_level_means_det) - wsjf_median
print(f"\nMedian difference vs RICE: {median_diff_rice:.2f} -> {'WITHIN' if abs(median_diff_rice) <= EQUIVALENCE_MARGIN else 'OUTSIDE'} equivalence margin (+/-{EQUIVALENCE_MARGIN})")
print(f"Median difference vs WSJF: {median_diff_wsjf:.2f} -> {'WITHIN' if abs(median_diff_wsjf) <= EQUIVALENCE_MARGIN else 'OUTSIDE'} equivalence margin (+/-{EQUIVALENCE_MARGIN})")

np.savez("frozen_protocol_results.npz",
         train_seeds=TRAIN_SEEDS, test_seeds=NEW_TEST_SEEDS,
         seed_level_means_det=seed_level_means_det, seed_level_means_stoch=seed_level_means_stoch,
         rice_per_scenario=rice_per_scenario, wsjf_per_scenario=wsjf_per_scenario,
         random_per_scenario=baseline_results["random"])
print("\nSaved frozen_protocol_results.npz")

Seed 10: deterministic mean = -1553.97 | stochastic mean =  -786.83
Seed 11: deterministic mean = -1050.41 | stochastic mean =  -489.02
Seed 12: deterministic mean = -1559.53 | stochastic mean =  -780.54
Seed 13: deterministic mean = -1182.99 | stochastic mean =  -525.33
Seed 14: deterministic mean = -1570.13 | stochastic mean =  -916.81
Seed 15: deterministic mean =  -380.07 | stochastic mean =  -382.56
Seed 16: deterministic mean = -1537.68 | stochastic mean =  -710.95
Seed 17: deterministic mean = -1569.11 | stochastic mean =  -864.36

PRIMARY RESULT: 8 seed-level means (deterministic)
Values: ['-1553.97', '-1050.41', '-1559.53', '-1182.99', '-1570.13', '-380.07', '-1537.68', '-1569.11']
Mean: -1300.49  Median: -1545.82
Std: 396.23  IQR: 412.09

Shapiro-Wilk normality test: statistic=0.7200, p=0.0038
-> Data is NOT normal (non-parametric test is appropriate)

--- Wilcoxon signed-rank test (primary, pre-registered) ---
vs RICE (median 19.64): W=0.000, raw p=0.0078, Holm-adjusted p=0.

In [5]:
print("Training convergence summary for all 8 seeds:\n")
for s in TRAIN_SEEDS:
    log = np.load(f"{MODEL_DIR}/convergence_seed{s}.npy")
    final_reward = log[-1][1]
    max_reward = log[:, 1].max()
    min_reward = log[:, 1].min()
    print(f"Seed {s}: final training reward = {final_reward:7.2f} | max = {max_reward:7.2f} | min = {min_reward:7.2f}")

Training convergence summary for all 8 seeds:

Seed 10: final training reward =   53.02 | max =   53.02 | min = -537.22
Seed 11: final training reward =   38.55 | max =   41.12 | min = -563.83
Seed 12: final training reward =   54.65 | max =   54.69 | min = -411.31
Seed 13: final training reward =   11.91 | max =   41.65 | min = -505.54
Seed 14: final training reward =   62.20 | max =   62.20 | min = -423.11
Seed 15: final training reward = -301.69 | max =   41.42 | min = -598.59
Seed 16: final training reward =   49.85 | max =   49.85 | min = -438.59
Seed 17: final training reward =   57.75 | max =   57.75 | min = -430.42


In [6]:
import torch

def get_now_probabilities(model, obs):
    """Extract each item slot's probability of choosing 'Now' (action=2) from the policy."""
    obs_tensor = torch.as_tensor(obs).float().unsqueeze(0)
    with torch.no_grad():
        distribution = model.policy.get_distribution(obs_tensor)
        # distribution.distribution is a list of Categorical distributions, one per item slot
        now_probs = [dist.probs[0, 2].item() for dist in distribution.distribution]
    return np.array(now_probs)

def run_episode_capacity_aware(model, seed):
    env = BacklogPrioritisationEnv(n_items=30, capacity_points=CAPACITY, seed=seed)
    obs, info = env.reset(seed=seed)
    rewards = []
    for cycle in range(N_CYCLES):
        now_probs = get_now_probabilities(model, obs)
        n_active = len(env.items)
        # Rank active items by the policy's own confidence in "Now"
        ranked_idx = np.argsort(-now_probs[:n_active])
        action = np.zeros(MAX_ITEMS, dtype=int)
        used_capacity = 0
        for idx in ranked_idx:
            item = env.items[idx]
            if used_capacity + item.effort_raw <= env.capacity_points:
                action[idx] = 2  # Now
                used_capacity += item.effort_raw
            elif now_probs[idx] > 0.3:
                action[idx] = 1  # Next (policy still wanted it, just no room)
            else:
                action[idx] = 0  # Later
        obs, reward, term, trunc, info = env.step(action)
        rewards.append(reward)
        if term or trunc:
            break
    return rewards

print("Testing capacity-aware decoding on all 8 frozen-protocol seeds:\n")
capacity_aware_means = []
for s in TRAIN_SEEDS:
    totals = [sum(run_episode_capacity_aware(ppo_models[s], ts)) for ts in NEW_TEST_SEEDS]
    capacity_aware_means.append(np.mean(totals))
    print(f"Seed {s}: capacity-aware mean = {np.mean(totals):8.2f}  (vs raw deterministic: {seed_level_means_det[TRAIN_SEEDS.index(s)]:8.2f})")

print(f"\nOverall: mean = {np.mean(capacity_aware_means):.2f}, median = {np.median(capacity_aware_means):.2f}")
print(f"RICE: {rice_median:.2f}  WSJF: {wsjf_median:.2f}")

Testing capacity-aware decoding on all 8 frozen-protocol seeds:

Seed 10: capacity-aware mean =    12.50  (vs raw deterministic: -1553.97)
Seed 11: capacity-aware mean =    11.95  (vs raw deterministic: -1050.41)
Seed 12: capacity-aware mean =    13.56  (vs raw deterministic: -1559.53)
Seed 13: capacity-aware mean =    11.63  (vs raw deterministic: -1182.99)
Seed 14: capacity-aware mean =    12.72  (vs raw deterministic: -1570.13)
Seed 15: capacity-aware mean =    10.58  (vs raw deterministic:  -380.07)
Seed 16: capacity-aware mean =    12.47  (vs raw deterministic: -1537.68)
Seed 17: capacity-aware mean =    12.27  (vs raw deterministic: -1569.11)

Overall: mean = 12.21, median = 12.37
RICE: 19.64  WSJF: 15.20


In [7]:
def explain_decision(item, tier, capacity_remaining):
    """Template-based rationale generator, tested here against genuine PPO decisions."""
    if tier == 2:  # Now
        if item.business_value > 0.6 and item.dependency_count == 0:
            return "high_value_no_dependency", "Prioritised for its high business value, with no blocking dependencies."
        elif item.business_value > 0.6 and item.dependency_count > 0:
            return "high_value_with_dependency", "Prioritised for its value, despite an unresolved dependency, flagged as a risk."
        elif item.urgency > 0.7:
            return "high_urgency", "Prioritised primarily due to its approaching urgency."
        elif item.demand_signal > 0.6:
            return "high_demand", "Prioritised due to strong stakeholder demand signal."
        else:
            return "generic_now", "Prioritised based on overall scoring across value, risk, and urgency."
    elif tier == 1:  # Next
        return "generic_next", "Scheduled for the next cycle; did not meet this cycle's priority threshold."
    else:  # Later
        return "generic_later", "Deferred; lower relative priority this cycle."

def check_fidelity(template_id, item):
    """Does the claimed reason actually hold true for this item? Returns True/False."""
    if template_id == "high_value_no_dependency":
        return item.business_value > 0.6 and item.dependency_count == 0
    elif template_id == "high_value_with_dependency":
        return item.business_value > 0.6 and item.dependency_count > 0
    elif template_id == "high_urgency":
        return item.urgency > 0.7
    elif template_id == "high_demand":
        return item.demand_signal > 0.6
    else:
        return None  # generic templates make no specific claim to verify

GENERIC_TEMPLATES = {"generic_now", "generic_next", "generic_later"}

def run_explainability_evaluation(model, seeds, n_cycles=12):
    total_decisions = 0
    specific_count = 0
    fidelity_checked = 0
    fidelity_correct = 0
    template_counts = {}

    for seed in seeds:
        env = BacklogPrioritisationEnv(n_items=30, capacity_points=CAPACITY, seed=seed)
        obs, info = env.reset(seed=seed)
        for cycle in range(n_cycles):
            now_probs = get_now_probabilities(model, obs)
            n_active = len(env.items)
            ranked_idx = np.argsort(-now_probs[:n_active])
            action = np.zeros(MAX_ITEMS, dtype=int)
            used_capacity = 0
            tier_by_idx = {}
            for idx in ranked_idx:
                item = env.items[idx]
                if used_capacity + item.effort_raw <= env.capacity_points:
                    tier_by_idx[idx] = 2
                    used_capacity += item.effort_raw
                elif now_probs[idx] > 0.3:
                    tier_by_idx[idx] = 1
                else:
                    tier_by_idx[idx] = 0
                action[idx] = tier_by_idx[idx]

            for idx, item in enumerate(env.items):
                tier = tier_by_idx.get(idx, 0)
                template_id, explanation_text = explain_decision(item, tier, env.capacity_points - used_capacity)
                total_decisions += 1
                template_counts[template_id] = template_counts.get(template_id, 0) + 1
                if template_id not in GENERIC_TEMPLATES:
                    specific_count += 1
                    fidelity_checked += 1
                    if check_fidelity(template_id, item):
                        fidelity_correct += 1

            obs, reward, term, trunc, info = env.step(action)
            if term or trunc:
                break

    coverage = specific_count / total_decisions * 100
    fallback_rate = 100 - coverage
    fidelity = (fidelity_correct / fidelity_checked * 100) if fidelity_checked > 0 else None

    print(f"Total decisions evaluated: {total_decisions}")
    print(f"Template coverage (specific, non-generic): {coverage:.1f}%")
    print(f"Generic fallback rate: {fallback_rate:.1f}%")
    print(f"Fidelity (specific explanations verified correct): {fidelity:.1f}% ({fidelity_correct}/{fidelity_checked})")
    print(f"\nTemplate distribution:")
    for tid, count in sorted(template_counts.items(), key=lambda x: -x[1]):
        print(f"  {tid}: {count} ({count/total_decisions*100:.1f}%)")
    print(f"\nUsefulness: NOT MEASURED -- requires human-participant evaluation, which has been formally withdrawn from this thesis's scope for ethics reasons (Ch. 6, Section 6.10).")

    return {"coverage": coverage, "fallback_rate": fallback_rate, "fidelity": fidelity, "n_decisions": total_decisions}

# Run it against your real, frozen-protocol trained policies
print("=== Explainability evaluation against genuine PPO decisions (frozen protocol seeds) ===\n")
all_results = {}
for s in TRAIN_SEEDS:
    print(f"--- Seed {s} ---")
    all_results[s] = run_explainability_evaluation(ppo_models[s], NEW_TEST_SEEDS)
    print()

import numpy as np
avg_coverage = np.mean([r["coverage"] for r in all_results.values()])
avg_fidelity = np.mean([r["fidelity"] for r in all_results.values() if r["fidelity"] is not None])
print(f"=== AVERAGE ACROSS ALL {len(TRAIN_SEEDS)} SEEDS ===")
print(f"Mean template coverage: {avg_coverage:.1f}%")
print(f"Mean fidelity: {avg_fidelity:.1f}%")

=== Explainability evaluation against genuine PPO decisions (frozen protocol seeds) ===

--- Seed 10 ---
Total decisions evaluated: 5400
Template coverage (specific, non-generic): 7.3%
Generic fallback rate: 92.7%
Fidelity (specific explanations verified correct): 100.0% (393/393)

Template distribution:
  generic_next: 4577 (84.8%)
  generic_now: 429 (7.9%)
  high_value_no_dependency: 181 (3.4%)
  high_value_with_dependency: 109 (2.0%)
  high_demand: 96 (1.8%)
  high_urgency: 7 (0.1%)
  generic_later: 1 (0.0%)

Usefulness: NOT MEASURED -- requires human-participant evaluation, which has been formally withdrawn from this thesis's scope for ethics reasons (Ch. 6, Section 6.10).

--- Seed 11 ---
Total decisions evaluated: 5400
Template coverage (specific, non-generic): 7.3%
Generic fallback rate: 92.7%
Fidelity (specific explanations verified correct): 100.0% (393/393)

Template distribution:
  generic_next: 4156 (77.0%)
  generic_later: 435 (8.1%)
  generic_now: 416 (7.7%)
  high_value_

In [8]:
import os
print(os.getcwd())

C:\Users\bhima\MR603_Frozen_Protocol
